# 02 — Clean & Quality Check

**Cleaning only** — type casts, column pruning, dedupe, light text normalization,
quality checks. Output: `data/interim/speeches_clean.parquet` + DuckDB
`speeches_clean`. **No feature engineering here** — pronoun metrics, speech-type
classification, word frequencies, TF-IDF, and the tenure table are all derived in
`03-prepare` (that's the analysis-ready / feature stage; this stage just makes the
raw data tidy and typed).

What this notebook does:
- parse the ISO `date` (bogus fixed `-04:56` offset) → `speech_date` + `year`;
- keep only the columns we use (drop `transcript_html`, `introduction`, `video`,
  `audio`, `uuid`);
- trim/standardize `president` and `title`; collapse transcript whitespace;
- drop empty-transcript rows and exact duplicates;
- quality report → save interim.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config
from src.clean_quality import (
    get_connection, run_sql, quality_report, save_interim, load_to_duckdb,
)

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## Clean in DuckDB: parse dates, prune columns, normalize text

The `date` field is ISO with a bogus fixed `-04:56` offset on every row — an
artifact — so we take the leading `YYYY-MM-DD`. We drop the unused columns, trim
whitespace on `president`/`title`, and collapse runs of whitespace in `transcript`
(a pure tidy — wording is untouched; HTML entities are decoded later at tokenization
time in prepare, not mutated here). Empty transcripts are dropped.

In [ ]:
df = run_sql("""
    SELECT
        TRIM(president)                              AS president,
        CAST(date[1:10] AS DATE)                     AS speech_date,
        CAST(date[1:4]  AS INTEGER)                  AS year,
        TRIM(title)                                  AS title,
        TRIM(regexp_replace(transcript, '\\s+', ' ', 'g')) AS transcript,
        url,
        source_file
    FROM speeches_raw
    WHERE transcript IS NOT NULL AND TRIM(transcript) <> ''
""", con)
# drop exact duplicate speeches (same president + date + title)
before = len(df)
df = df.drop_duplicates(subset=['president','speech_date','title']).reset_index(drop=True)
print(f'rows: {before} → {len(df)} after dedupe')
print('year span:', int(df.year.min()), '→', int(df.year.max()))
print('columns:', list(df.columns))
df.head(3)

## Quality report + save interim

Check the cleaned table, then persist to `data/interim/` and DuckDB `speeches_clean`.
`03-prepare` reads this to build the feature tables.

In [ ]:
qr = quality_report(
    df, table_name='speeches_clean', con=con,
    required_columns=['president','speech_date','year','title','transcript'],
    max_null_pct=0.02,
)

In [ ]:
load_to_duckdb(df, 'speeches_clean', con)
save_interim(df, cfg, 'speeches_clean.parquet')
con.execute('DROP TABLE IF EXISTS _qc_speeches_clean')  # drop QC scratch table
print('speeches_clean rows in DuckDB:',
      con.execute('SELECT COUNT(*) FROM speeches_clean').fetchone()[0])

---
**Next:** `03-prepare.ipynb` — FEATURE ENGINEERING: derive pronoun metrics +
speech-type (`speeches_features`), president tenure table, and word-frequency /
TF-IDF tables; then package the export + codebook.

---
## Cleanup
Close the DuckDB connection so the lock is released for other tools (DBCode, other notebooks). Runs on “Run All”.

In [ ]:
con.close()
print('connection closed')